# Modular semi-Mapper pipeline — CTN-0051

This notebook is a thin driver over the `mapper` package. All logic lives in the package modules; here you only **tune parameters and call the pipeline**.

Pipeline: `data -> distance -> lens -> cover -> graph -> layout -> viz`.

Three lenses are available:
- **feature**    — a data column (original behaviour, e.g. `attendance_density_w8`)
- **centrality** — graph centrality on the proximity graph (degree, betweenness, ...)
- **density**    — local density in feature space (knn, ball, kde)

## 1. Parameters — tune here

In [4]:
from mapper import MapperParams, run_pipeline, visualise
from mapper import diagnostics as dg
import numpy as np

params = MapperParams(
    PRE_TRIAL_CSV="../../data/clean-data/pre-trial.csv",
    TARGET_CSV   ="../../data/clean-data/retention_tier.csv",
    W8_CSV       ="../../data/clean-data/features_w8.csv",
    W12_CSV      ="../../data/clean-data/features_w12.csv",
    W24_CSV      ="../../data/clean-data/features_w24.csv",

    # --- proximity ---
    EPSILON=0.4
    ,
    METRIC ="cosine",        # euclidean | cosine | manhattan | minkowski
    MINKOWSKI_P= np.inf,            # only for minkowski

    # --- LENS: pick one of the three ---
    LENS_KIND="feature",        # "feature" | "centrality" | "density"
    FEATURE_LENS_COL  = "detox_los",   # feature lens: attendance_density_w8 | detox_los 
    CENTRALITY_MEASURE="betweenness",             # centrality lens

    DENSITY_METHOD    ="kde",                     # density lens
    DENSITY_K         =10,

    # --- COVER / BINNING (tunable) ---
    COVER_MODE ="balanced",      # "uniform" (N_INTERVALS+OVERLAP) or "edges" (BIN_EDGES)
    N_INTERVALS= 100,
    OVERLAP    =0.5,
    # For explicit clinical bins instead, use:
    # COVER_MODE="edges", BIN_EDGES=[0,3,7,14,21], BIN_LABELS=["0-3","3-7","7-14","14+"]

    PIECEWISE_SEGMENTS=[
        (350, 360, 15),    # dense cover in the first half
        (12, 40, 5),
         # sparse cover in the second half
    ],



    # --- edge rule for the displayed graph ---
    EDGE_RULE  ="cover",        # "cover" (share a set) | "gap" | "none"
    MAX_BIN_GAP=1,

    # --- layout & colour ---
    LAYOUT  ="spring", 
    SPRING_K = 1,              # spring | spectral | pca
    COLOR_BY="tier",            # "tier" or "lens"
)
print(params.summary())

ε=0.4 | metric=cosine | lens=feature=detox_los | cover=edges | edge_rule=cover | layout=spring


## 2. Run the pipeline

In [5]:
result = run_pipeline(params)   # prints stage-by-stage summaries

Patients        : 554
Feature matrix  : (554, 19)
Missing values  : 0

Features (19):
  last_opioid_date
  last_substance_code
  last_route_code
  discharge_facility_code
  detox_los
  education_years
  occupation_category
  has_drivers_licence
  has_automobile
  days_paid_employment_30d
  days_any_employment_30d
  income_employment_30d
  income_unemployment_30d
  income_welfare_30d
  income_illegal_30d
  n_dependants
  had_relapse_by_w8
  engagement_onset_w8
  attendance_density_w8

Retention tier distribution:
retention_tier
1    164
2    108
3     61
4    221 

Distance matrix : (554, 554)
Distance range  : [0.000, 1.864]
Percentiles:
   10th : 0.605
   25th : 0.809
   50th : 1.020
   75th : 1.205
   90th : 1.357
With ε = 0.4:
  Edges before pruning : 4084
  Edge density         : 2.7% 

[feature] detox_los: min=1.000 median=6.000 max=40.000 (n_valid=554) 

Cover mode: balanced  |  25 sets

  Set  0 [0.735,1.79)    :   6 patients
  Set  1 [1.29,2.23)     :  23 patients
  Set  2 [1.5

## 3. Visualization

In [6]:
from bokeh.io import output_notebook
output_notebook()
visualise(result)

Loading BokehJS ...

figure(id='p1112', ...)

In [4]:
from bokeh.io import output_file, save
import networkx as nx
import copy

# Get the figure without rendering it in the notebook
fig = visualise(result, render=False)

# Save Bokeh HTML
output_file("../../graphs/w24/b/KDE_b.html", title="Metric Graph – Week 24 Attendance Density")
save(fig)

# GraphML only supports scalar types — convert any list attributes to strings
G_export = copy.deepcopy(result.graph)
for _, data in G_export.nodes(data=True):
    for k, v in list(data.items()):
        if isinstance(v, list):
            data[k] = ",".join(map(str, v))
for _, _, data in G_export.edges(data=True):
    for k, v in list(data.items()):
        if isinstance(v, list):
            data[k] = ",".join(map(str, v))

nx.write_graphml(G_export, "../../graphs/w24/b/KDE_b.graphml")
# Reload later:
G = nx.read_graphml("../../graphs/w24/b/KDE_b.graphml")